Hier werden die Datenverträge, die Abbildung der Quellen auf das interne Format sowie die Beschaffung und Normalisierung der Untersuchungsdaten dokumentiert. Die allgemeine Planung wird von den anschliessenden Angaben zur konkreten Studie unterschieden. Die für den Vergleich wesentlichen Datenentscheidungen bleiben in Methodik und Abschnitt 5.2 beschrieben.



## Entwicklung eines einheitlichen Datenformats {#sec-daten-5-3}



### Warum überhaupt ein eigenes internes Format? {#sec-daten-5-3-1}


<p>Die später verwendeten Daten können aus unterschiedlichen Quellen stammen. Yahoo Finance liefert Marktzeitreihen anders als FRED eine Zinsreihe oder eine Makrodatenbank jährliche BIP-Werte. Würde jede Strategie direkt mit dem Format einer bestimmten Quelle arbeiten, müsste sich auch die Strategie ändern, sobald der Anbieter gewechselt wird.</p>
<p>Deshalb erhält das Modell einen <strong>anbieterunabhängigen Datenvertrag</strong>. Ein Importmodul ist dafür verantwortlich, Quelldaten in diesen Vertrag zu übersetzen. Die eigentliche Strategie sieht danach nur noch standardisierte Daten.</p>
<p>Wes McKinney beschreibt Datenanalyse als Abfolge von Einlesen, Bereinigen, Kombinieren, Normalisieren, Umformen und anschliessender Berechnung. Er behandelt dabei sowohl tabellarische Daten als auch Zeitreihen und zeigt, dass pandas gerade für solche strukturierten Daten entwickelt wurde [@mckinneyPythonDataAnalysis2022]. Diese Grundidee passt zu meiner Untersuchung: Die Quelle darf unterschiedlich sein, die Berechnung soll danach aber auf einer einheitlichen Struktur stattfinden.</p>
<p>Die Entscheidung, dafür einen eigenen Datenvertrag zu definieren, ist meine technische Schlussfolgerung aus diesem Bedarf.</p>


### Warum CSV als Austauschformat? {#sec-daten-5-3-2}


<p>Für den Austausch zwischen Import, Kontrolle und späterer Benutzeroberfläche verwende ich in der ersten Version CSV-Dateien. McKinney behandelt in Kapitel 6 das Einlesen tabellarischer Textdaten und verwendet <code>pandas.read_csv()</code> als eines der zentralen Werkzeuge für solche Daten [@mckinneyPythonDataAnalysis2022].</p>
<p>CSV ist für mein Projekt nicht deshalb ausgewählt, weil es allen anderen Formaten technisch überlegen wäre. Die Gründe sind praktischer:</p>

- Die Datei kann direkt mit einem Texteditor, Tabellenprogramm oder Python geöffnet werden.
- Beim Entwickeln kann ich leicht kontrollieren, ob ein Adapter tatsächlich die erwarteten Spalten erzeugt.
- Eine spätere Weboberfläche kann eine CSV-Datei einfach als Benutzereingabe akzeptieren.
- pandas kann das Format ohne zusätzliche Spezialbibliothek einlesen.

<p>CSV besitzt gleichzeitig Grenzen. Datentypen und die Bedeutung einer Spalte werden nicht ausreichend durch die Datei selbst erklärt. Darum genügt eine CSV allein nicht: Das System benötigt zusätzlich einen klar definierten Datenvertrag und Metadaten.</p>
<p>Für die interne Berechnung bleibt die Datei nicht als Textstruktur bestehen. Nach dem Einlesen wird sie in pandas-Objekte überführt.</p>



### Warum ein Long-Format? {#sec-daten-5-3-3}


<p>Für die Marktzeitreihen verwende ich als Austauschformat eine lange Tabelle. Eine Beobachtung besteht aus einem Datum, einer Anlagenkennung und einem Wert.</p>
<p>McKinney beschreibt im Abschnitt "Pivoting 'Long' to 'Wide' Format" ausdrücklich, dass mehrere Zeitreihen häufig in einem sogenannten <i>long</i> beziehungsweise <i>stacked format</i> gespeichert werden. In diesem Format stellt jede einzelne Beobachtung eine eigene Zeile dar. Er zeigt danach mit <code>DataFrame.pivot()</code>, wie dieselben Daten für Berechnungen in eine breite Tabelle mit einer Spalte pro Zeitreihe umgeformt werden können [@mckinneyPythonDataAnalysis2022].</p>
<p>Diese Beschreibung passt direkt zu meinem Problem. Für ein allgemeines Eingabeformat ist Long praktisch, weil eine neue Anlage zusätzliche Zeilen erzeugt, ohne dass sich die Grundstruktur der Tabelle ändert. Für bestimmte Berechnungen ist dagegen eine breite Darstellung komfortabler. Deshalb lege ich nicht fest, dass jede Stufe des Systems Long sein muss:</p>

$$
\begin{array}{c}
\mathbf{\text{Austausch / Speicherung:}}
\\[4pt]
\boxed{\texttt{date} \;|\; \texttt{asset\_id} \;|\; \texttt{performance\_value}}
\\[10pt]
\downarrow
\\[-2pt]
\texttt{pandas pivot()}
\\[10pt]
\downarrow
\\[-2pt]
\mathbf{\text{Interne Berechnung:}}
\\[4pt]
\boxed{\texttt{date} \;|\; \texttt{ASSET\_A} \;|\; \texttt{ASSET\_B} \;|\; \texttt{ASSET\_C} \;|\; \ldots}
\end{array}
$$

<p>McKinney liefert die nachvollziehbare Grundlage dafür, dass Long ein gebräuchliches Format für mehrere Zeitreihen ist und sich mit pandas eindeutig in Wide umformen lässt. Meine Entscheidung ist, Long deshalb als äussere Schnittstelle und Wide dort zu verwenden, wo die Berechnung davon profitiert.</p>


### Marktzeitreihen: minimaler Datenvertrag {#sec-daten-5-3-4}



<p>Für eine Marktzeitreihe definiere ich zunächst drei Pflichtfelder:</p>

$$
\begin{array}{|l|l|}
\hline
\textbf{Feld} & \textbf{Bedeutung} \\
\hline
\texttt{date} & \text{Datum der Beobachtung} \\
\hline
\texttt{asset\_id} & \text{interne, anbieterunabhängige Kennung der Anlage} \\
\hline
\texttt{performance\_value} & \text{Wertreihe, aus der die Performance berechnet werden darf} \\
\hline
\end{array}
$$

<p>Optional kann eine vierte Spalte vorhanden sein:</p>

$$
\begin{array}{|l|l|}
\hline
\textbf{Feld} & \textbf{Bedeutung} \\
\hline
\texttt{signal\_value} & \text{separate Wertreihe für technische Handelssignale} \\
\hline
\end{array}
$$

<p>Beispiel:</p>

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{date} & \textbf{asset\_id} & \textbf{performance\_value} & \textbf{signal\_value} \\
\hline
2020\text{-}01\text{-}02 & \texttt{EQ\_DEMO} & 100.0 & 50.25 \\
\hline
2020\text{-}01\text{-}03 & \texttt{EQ\_DEMO} & 102.0 & 51.00 \\
\hline
2020\text{-}01\text{-}02 & \texttt{BD\_DEMO} & 100.0 & 104.80 \\
\hline
2020\text{-}01\text{-}03 & \texttt{BD\_DEMO} & 100.5 & 104.90 \\
\hline
\end{array}
$$

<p><code>asset_id</code> ist bewusst nicht dasselbe wie ein Yahoo-Ticker. Der Ticker gehört zur Quelle; <code>asset_id</code> gehört zum Modell. Dadurch könnte dieselbe logische Anlage später aus einer anderen Quelle importiert werden, ohne dass alle Strategiekonfigurationen angepasst werden müssen.</p>

<p>Auch <code>performance_value</code> ist absichtlich kein Anbieterbegriff wie <code>Adj Close</code>. Der Importadapter muss dokumentieren, welche Quellenspalte auf dieses Feld abgebildet wurde.</p>

In [1]:
from io import StringIO
import pandas as pd

beispiel_csv = "\n".join([
    "date,asset_id,performance_value",
    "2020-01-02,EQ_DEMO,100.0",
    "2020-01-03,EQ_DEMO,102.0",
    "2020-01-06,EQ_DEMO,101.0",
    "2020-01-02,BD_DEMO,100.0",
    "2020-01-03,BD_DEMO,100.5",
    "2020-01-06,BD_DEMO,101.0",
])

markt_long = pd.read_csv(StringIO(beispiel_csv))
markt_long["date"] = pd.to_datetime(markt_long["date"])
markt_long

,date,asset_id,performance_value
0,2020-01-02,EQ_DEMO,100.0
1,2020-01-03,EQ_DEMO,102.0
2,2020-01-06,EQ_DEMO,101.0
3,2020-01-02,BD_DEMO,100.0
4,2020-01-03,BD_DEMO,100.5
5,2020-01-06,BD_DEMO,101.0


<p>Für die spätere Portfolioberechnung kann dieselbe Tabelle eindeutig umgeformt werden. Dadurch muss die externe Datei nicht bereits für jede konkrete mathematische Funktion optimiert sein.</p>

In [4]:
markt_wide = (
    markt_long
    .pivot(index="date", columns="asset_id", values="performance_value")
    .sort_index()
    .reset_index()
)
    
markt_wide.columns.name = None
markt_wide

,date,BD_DEMO,EQ_DEMO
0,2020-01-02,100.0,100.0
1,2020-01-03,100.5,102.0
2,2020-01-06,101.0,101.0



### Was bedeutet `performance_value`? {#sec-daten-5-3-5}



<p>Die Spalte <code>performance_value</code> ist die Wertreihe, aus der das Modell später die tatsächliche Wertentwicklung einer Anlage berechnet. Dabei soll nicht einfach irgendein verfügbarer Kurs übernommen werden. Für meine Untersuchung ist entscheidend, dass die gewählte Reihe zur festgelegten Behandlung der Dividenden passt.</p>

<p>Im Theorieteil wurde bereits festgelegt, dass für den langfristigen Vergleich eine Gesamtrendite beziehungsweise ein <i>Total Return</i> geeigneter ist als eine reine Kursbetrachtung. Ein Total Return berücksichtigt neben Kursveränderungen auch laufende Erträge wie Dividenden [@kommerSouveraenInvestierenMit2025, S. 466–467]. Da in meiner Methodik die Reinvestition der Dividenden vorgesehen ist, muss die verwendete Performance-Reihe diese Annahme abbilden.</p>

<p><code>performance_value</code> bezeichnet deshalb bewusst keinen bestimmten Anbieterbegriff. Je nach Datenquelle kann dahinter zum Beispiel ein Total-Return-Index oder eine entsprechend bereinigte Wertreihe stehen. Der jeweilige Import muss dokumentieren, welche ursprüngliche Reihe verwendet wurde.</p>

<p>Dadurch bleibt die Berechnung unabhängig von der Quelle. Die Engine erhält immer dieselbe Spalte, während die Umwandlung davor sicherstellt, dass ihre Bedeutung vergleichbar ist.</p>


### Wozu dient `signal_value`? {#sec-daten-5-3-6}



<p>Für Buy-and-Hold oder ein Rebalancing-Portfolio genügt grundsätzlich die Performance-Reihe. Bei einer technischen Strategie wie der Trendfolge kommt jedoch eine zweite Aufgabe hinzu: Aus einer Kursreihe wird ein Handelssignal berechnet.</p>
<p>Hilpisch verwendet für seine SMA-Strategie zwei gleitende Durchschnitte und leitet daraus eine Long- beziehungsweise Short-Position ab [@hilpischPythonFinanceMastering2019, S. 691–692]. Für meine Untersuchung wurde diese Regel bereits im Theorieteil zu Long/Cash angepasst. Ausserdem wird das Signal um eine Periode verschoben, damit keine bereits bekannte Rendite derselben Periode verwendet wird [@hilpischPythonFinanceMastering2019, S. 694].</p>
<p>Damit das allgemeine System später nicht voraussetzen muss, dass Performance-Berechnung und Signal immer auf exakt derselben Reihe beruhen, kann zusätzlich <code>signal_value</code> angegeben werden. Fehlt diese Spalte, darf die Strategie <code>performance_value</code> nur dann verwenden, wenn dies in der Konfiguration ausdrücklich erlaubt ist.</p>
<p>Für die erste Untersuchung kann sich später herausstellen, dass beide Felder auf dieselbe Reihe zeigen. Die Trennung bleibt trotzdem sinnvoll, weil sie die Datenstruktur nicht auf diese eine Untersuchung beschränkt.</p>


### Metadaten und zusätzliche Datentypen {#sec-daten-5-3-7}



<p>Die Marktdatei enthält absichtlich nur die Werte, die sich über die Zeit verändern. Weitere Informationen zu einer Anlage werden separat gespeichert. Dazu gehören beispielsweise Name, Anlageklasse, Währung, Land und ursprüngliche Datenquelle.</p>

<p>Für diese Informationen ist eine Datei <code>assets.csv</code> vorgesehen:</p>

$$
\begin{array}{c|c}
\textbf{Feld} & \textbf{Bedeutung} \\
\hline
\texttt{asset\_id} & \text{Interne, anbieterunabhängige Kennung der Anlage} \\
\texttt{name} & \text{Name der Anlage} \\
\texttt{asset\_class} & \text{Anlageklasse, z. B. Aktien oder Anleihen} \\
\texttt{country} & \text{Zugeordnetes Land bzw. Markt} \\
\texttt{currency} & \text{Währung der Anlage} \\
\texttt{provider} & \text{Datenanbieter} \\
\texttt{provider\_symbol} & \text{Symbol bzw. Kennung beim Datenanbieter}
\end{array}
$$

<p>Die interne Kennung <code>asset_id</code> verbindet die Metadaten mit den Marktwerten. Dadurch kann zum Beispiel <code>US_EQUITY</code> im Modell gleich bleiben, auch wenn die Daten später von einem anderen Anbieter bezogen werden.</p>

<p>Nicht alle benötigten Daten sind Marktpreise. Für die Sharpe Ratio wird eine risikofreie Rendite benötigt. Sie muss in derselben Periodenlogik wie die untersuchte Rendite verwendet werden [@kommerSouveraenInvestierenMit2025, S. 165–166]. Dafür ist eine getrennte Datei vorgesehen:</p>

$$
\begin{array}{c|c}
\textbf{Feld} & \textbf{Bedeutung} \\
\hline
\texttt{date} & \text{Datum der Beobachtung} \\
\texttt{series\_id} & \text{Kennung der risikofreien Zinsreihe} \\
\texttt{value} & \text{Beobachteter Zinssatz} \\
\texttt{unit} & \text{Einheit des Wertes}
\end{array}
$$

<p>Auch Makrodaten wie das BIP unterscheiden sich von täglichen Marktzeitreihen. Sie werden deshalb separat gespeichert:</p>

$$
\begin{array}{c|c}
\textbf{Feld} & \textbf{Bedeutung} \\
\hline
\texttt{period} & \text{Zeitraum der Beobachtung} \\
\texttt{country} & \text{Land, auf das sich der Wert bezieht} \\
\texttt{indicator} & \text{Makroökonomischer Indikator} \\
\texttt{value} & \text{Beobachteter Wert} \\
\texttt{unit} & \text{Einheit des Wertes} \\
\texttt{available\_from} & \text{Zeitpunkt, ab dem die Information verfügbar war}
\end{array}
$$

<p>Das zusätzliche Feld <code>available_from</code> soll verhindern, dass ein historischer Backtest einen Jahreswert bereits verwendet, bevor dieser damals verfügbar gewesen wäre. Die genaue Regel dafür wird später bei der BIP-Strategie festgelegt. Sie folgt dem gleichen Grundprinzip wie die im Theorieteil behandelte Vermeidung von Look-ahead Bias [@hilpischPythonFinanceMastering2019, S. 319–321].</p>


## Datenquellen und Umwandlung {#sec-daten-5-4}



### Trennung zwischen Datenquelle und Modell {#sec-daten-5-4-1}



<p>Nachdem das interne Format festgelegt ist, kann untersucht werden, wie verschiedene Anbieter darauf abgebildet werden. Die Datenquelle soll dabei nicht direkt von einer Strategie aufgerufen werden. Stattdessen übernimmt ein Importmodul die Umwandlung.</p>

$$
\begin{array}{c}
\boxed{\text{Datenquelle}} \\[6pt]
\downarrow \\[6pt]
\boxed{\text{Import und Umwandlung}} \\[6pt]
\downarrow \\[6pt]
\boxed{\text{Einheitliches Datenformat}} \\[6pt]
\downarrow \\[6pt]
\boxed{\text{Simulation}}
\end{array}
$$

<p>Dadurch bleibt die eigentliche Berechnung gleich, auch wenn später eine andere Datenquelle verwendet wird. Für meine Untersuchung sind Yahoo Finance und FRED bereits in der Methodik als mögliche Quellen vorgesehen [@yahooYahooFinanzenDeutschlandBoerse2026; @federalreservebankofst.louisFederalReserveEconomic2026]. Weitere Quellen können später ergänzt werden, solange sie auf dasselbe interne Format abgebildet werden können.</p>


### Marktdaten {#sec-daten-5-4-2}




<p>Für Marktzeitreihen soll ein erster Adapter historische Daten eines Anbieters laden und daraus die benötigten Felder erzeugen. Bei Yahoo Finance beziehungsweise einer Python-Schnittstelle wie <code>yfinance</code> muss der Adapter insbesondere festhalten:</p>

- welches Wertpapier geladen wurde,
- welcher Zeitraum verwendet wurde,
- welche Datenfrequenz angefordert wurde,
- welche ursprüngliche Spalte zu <code>performance_value</code> wird,
- welche Reihe für <code>signal_value</code> verwendet wird,
- und wie Dividenden beziehungsweise Splits in den Quelldaten behandelt werden.

<p>Die technische Umbenennung einer Spalte reicht damit nicht aus. Zuerst muss geklärt sein, was die ursprüngliche Reihe tatsächlich bedeutet. Für konkrete Parameter einer später verwendeten API soll bei der Implementierung zusätzlich deren direkte technische Dokumentation verwendet werden.</p>


### Zins- und Makrodaten {#sec-daten-5-4-3}



<p>FRED eignet sich als mögliche Quelle für Zinsreihen und andere wirtschaftliche Daten. Für eine risikofreie Reihe ist besonders wichtig, die Einheit der Quelle zu übernehmen. Ein als jährlicher Prozentsatz veröffentlichter Zinssatz darf nicht unverändert von einer täglichen Rendite abgezogen werden. Die Umrechnung muss deshalb erst erfolgen, nachdem Frequenz und Einheit bekannt sind.</p>

<p>Für die spätere BIP-Gewichtung wird zusätzlich eine Quelle mit historischen Länderwerten benötigt. Diese Daten werden zunächst in das allgemeine Makroformat überführt. Welche konkrete Datenbank, welche Länder und welche Veröffentlichungsverzögerung in der Hauptuntersuchung verwendet werden, wird erst bei der endgültigen Versuchskonfiguration festgelegt.</p>

<p>Damit bleibt an dieser Stelle noch offen, welche konkrete Serie später verwendet wird. Festgelegt ist nur, welche Informationen das Modell benötigt.</p>


### Import eigener CSV-Dateien {#sec-daten-5-4-4}



<p>Damit das System auch ausserhalb meiner später ausgewählten Anbieter verwendet werden kann, soll zusätzlich ein allgemeiner CSV-Import möglich sein. Dabei kann eine fremde Datei über eine kleine Zuordnung auf das interne Format abgebildet werden.</p>

<p>Beispiel einer Quelldatei:</p>

$$
\begin{array}{c|c|c}
\textbf{Date} & \textbf{Ticker} & \textbf{TR} \\
\hline
\texttt{2020-01-02} & \texttt{ABC} & 100.0
\end{array}
$$

<p>Dazu könnte folgende Zuordnung gehören:</p>

$$
\begin{aligned}
\texttt{Date}   &\;\longrightarrow\; \texttt{date} \\
\texttt{Ticker} &\;\longrightarrow\; \texttt{provider\_symbol} \\
\texttt{TR}     &\;\longrightarrow\; \texttt{performance\_value}
\end{aligned}
$$

<p>Der Import kann damit unterschiedliche Spaltennamen vereinheitlichen. Er kann jedoch nicht automatisch beurteilen, ob die Werte fachlich für eine bestimmte Untersuchung geeignet sind. Diese Prüfung bleibt Teil der Datenwahl und Dokumentation.</p>


## Datenaufbereitung und Qualitätskontrolle {#sec-daten-5-5}



### Rohdaten und verarbeitete Daten {#sec-daten-5-5-1}



<p>Die ursprünglich geladenen Rohdaten und die später tatsächlich verwendeten Daten werden getrennt gespeichert:</p>

$$
\begin{array}{l}
\texttt{data/} \\
\quad \texttt{|-- raw/} \\
\quad \texttt{`-- processed/}
\end{array}
$$

<p>Diese Trennung erleichtert die Nachvollziehbarkeit. Wenn sich eine Regel bei der Aufbereitung ändert, kann sie erneut auf denselben Rohdaten ausgeführt werden. Die Originaldatei muss dafür nicht manuell verändert werden.</p>

<p>McKinney unterscheidet beim Arbeiten mit Daten ebenfalls zwischen Einlesen, Bereinigen, Umformen und anschliessender Analyse [@mckinneyPythonDataAnalysis2022, Kap. 6–8]. Die konkrete Ordnerstruktur ist an mein Projekt angepasst.</p>


### Datenfrequenz und Resampling {#sec-daten-5-5-4}



<p>Die Theorie hat bereits gezeigt, dass Finanzzeitreihen in unterschiedlichen Frequenzen vorliegen können. Für das Downsampling einer Wertreihe wird der letzte verfügbare Wert eines abgeschlossenen Intervalls verwendet. Die zeitlich korrekte Zuordnung ist wichtig, damit keine zukünftige Information in eine frühere Periode gelangt [@hilpischPythonFinanceMastering2019, S. 319–321].</p>
<p>Dabei muss zwischen Kurs- beziehungsweise Wertreihen und bereits berechneten Renditen unterschieden werden. Wird beispielsweise aus Tageswerten eine Monatsreihe erzeugt, kann der letzte Wert des Monats verwendet werden. Mehrere Tagesrenditen eines Monats dürfen dagegen nicht einfach durch die letzte Tagesrendite ersetzt werden; sie müssen über ihre Wachstumsfaktoren miteinander verknüpft werden.</p>
<p>Das vorhandene <code>resample_dataframe()</code> bildet damit einen Teil der späteren Verarbeitung ab. Bei der Integration muss geprüft werden, für welche Datentypen die jeweilige Aggregationsregel zulässig ist.</p>

<p>Die Engine verlangt für ihre Jahreskennzahlen eine nachgewiesene Periodenlogik. Für diese vorbereiteten Monatsdaten werden <code>period_frequency="ME"</code> und <code>periods_per_year=12</code> gesetzt. Diese Festlegung betrifft zwölf Monatsintervalle pro Jahr und keine vermeintlich täglich handelbare Reihe. Die tatsächlichen Quelltage bleiben im Aufbereitungsprotokoll sichtbar.</p>

### Technische Zeitreihen- und Resampling-Beispiele {#sec-resampling-beispiele}

Diese Originalangaben dokumentieren die technische Einführung aus dem Theoriekapitel. Die Aliasliste und ihre Quellenbelege bleiben im historischen Kontext erhalten; die vorhandene Python-Codezelle und ihre gespeicherte Ausgabe verbleiben im Theoriekapitel.

Die folgende historische Aliasübersicht stammt aus Hilpisch (2019). Neuere pandas-Versionen verwenden unter anderem [ME statt M für das Monatsende](https://pandas.pydata.org/docs/whatsnew/v2.2.0.html#deprecate-aliases-m-q-y-etc-in-favour-of-me-qe-ye-etc-for-offsets). Der dokumentierte Studienstand nutzt pandas 2.3.3 und ME.


Für diese Zeitreihen wird pandas in Python sehr wichtig sein. Wes McKinney, der Hauptautor von pandas, arbeitete als Analyst bei AQR Capital Management, als er mit der Entwicklung der Library begann. Man kann also sicher sagen, dass pandas darauf ausgelegt ist, mit Finanzdaten zu arbeiten [@hilpischPythonFinanceMastering2019, S. 304].

| Alias | Description |
|:---|:---|
| `B` | Business day frequency |
| `C` | Custom business day frequency (experimental) |
| `D` | Calendar day frequency |
| `W` | Weekly frequency |
| `M` | Month end frequency |
| `BM` | Business month end frequency |
| `MS` | Month start frequency |
| `BMS` | Business month start frequency |
| `Q` | Quarter end frequency |
| `BQ` | Business quarter end frequency |
| `QS` | Quarter start frequency |
| `BQS` | Business quarter start frequency |
| `A` | Year end frequency |
| `BA` | Business year end frequency |
| `AS` | Year start frequency |
| `BAS` | Business year start frequency |
| `H` | Hourly frequency |
| `T` | Minutely frequency |
| `S` | Secondly frequency |
| `L` | Milliseconds |
| `U` | Microseconds |

<p>Ausführliche Liste mit Beispielfrequenzen [@hilpischPythonFinanceMastering2019, S. 191-192].</p>


<code><pre>result = df.resample(frequenz).agg(aggregation)</pre></code>

<p>Für das "Resampling" gibt es in pandas eine eingebaute Funktion, die oben stark vereinfacht dargestellt ist. Nun folgt ein konkretes Beispiel für die Nutzung dieser Funktion.</p>

Quelle zur Nutzung von to_datetime() [@mckinneyPythonDataAnalysis2022, Kapitel 11, «Converting Between String and Datetime»].


## Beschaffung und unveränderte Ablage {#sec-daten-5-11-1}



<p>Marktreihen werden über die bereits im Projekt vorgesehene Bibliothek <code>yfinance</code> bezogen. Der Aufruf legt Tagesfrequenz, Anfangs- und Enddatum sowie die Adjustierungsoptionen ausdrücklich fest. <code>auto_adjust=False</code> hält <code>Close</code> und <code>Adj Close</code> getrennt; <code>repair=False</code> verhindert die zusätzlich angebotene automatische Reparatur. Ausschüttungs- und Splitspalten werden mitgespeichert [@PriceHistoryClassYfinance]. Der gespeicherte Rohstand ist ein unveränderter Bibliotheksexport, nicht die Behauptung eines archivierten ursprünglichen HTTP-Antwortpakets.</p>

<p>Für VT und die sechs nichtamerikanischen Länder-ETFs wird der Zeitraum vom 1. Januar 2009 bis zum 31. Dezember 2025 angefordert. Dadurch stehen für die Hauptuntersuchung genügend Monatsbeobachtungen für den SMA-Vorlauf zur Verfügung, bevor die eigentliche Auswertung 2010 beginnt. Für VTI und IEF beginnt der Download bereits am 1. Januar 2002. VTI benötigt die Beobachtungen aus dem Jahr 2002 für den SMA-Vorlauf, während IEF ab Mitte 2002 verfügbar ist und somit für die Startbewertung Ende Dezember 2002 berücksichtigt werden kann. Die eigentliche längere US-Untersuchung beginnt anschliessend im Jahr 2003. DGS3MO wird entsprechend für 2002 bis 2025 separat gespeichert, damit auch für die erste Renditeperiode des US-Zusatzlaufs eine passende risikofreie Reihe vorbereitet werden kann. Die WEO-Dateien bleiben im originalen Downloadformat; ein Öffnen und erneutes Speichern in Excel ist nicht Teil des Rohdatenarchivs.</p>

<p>Zu jedem Download werden Quelle, Parameter, Abrufzeit in UTC, tatsächliche Zeilenabdeckung und SHA-256 gespeichert. Frühere Downloads werden nicht überschrieben. Die Rohdaten liegen getrennt von bearbeiteten Dateien und Ergebnisdateien. Vor einer öffentlichen Weitergabe wird die Zulässigkeit der jeweiligen Datenweiterverwendung geprüft.</p>

### Dokumentierter Vorprüfstand der WEO-Ausgaben {#sec-weo-vorpruefung}

Die folgende Passage hält den ursprünglichen Planungsstand vor der vollständigen Beschaffung fest. Sie wird als historischer Verfahrensnachweis übernommen; die durchgeführte Eingangsprüfung steht in @sec-weo-eingang.


<p>Die vollständige historische Datenabdeckung ist vor dem Hauptlauf noch zu bestätigen. Deshalb werden zuerst die Dateien von 2009 und 2024 geprüft und erst danach die dazwischenliegenden Ausgaben beschafft. Dieser Schritt prüft das Format und die historische Nachvollziehbarkeit, nicht die Rendite der späteren BIP-Strategie.</p>


## Normalisierung und Datenkontrolle {#sec-daten-5-11-2}



<p>Die Datenaufbereitung arbeitet ausschliesslich auf dem gespeicherten Rohstand. Sie prüft Datumswerte, numerische Werte, Duplikate, Metadaten, Währung und Monatsabdeckung. Aus den täglichen Marktbeobachtungen entsteht die festgelegte Monatsreihe. Ein zusätzlicher Herkunftsnachweis erhält die tatsächlichen Handelstage zu den Monatslabels. Leere Monate, unterschiedliche letzte Monatshandelstage oder unplausible Abstände werden nicht durch stilles Füllen verborgen.</p>

<p>Der Zinsadapter erstellt normalisierte RF-Perioden. Der WEO-Adapter unterscheidet Datenversion, Bezugsjahr und Verfügbarkeit und dokumentiert jeden Ausschluss von Schätzungen oder Prognosen. Für die Länder- und Metadatenkennungen werden dieselben ISO-Ländercodes verwendet. Fehlerhafte Eingaben führen vor dem Hauptlauf zu einem nachvollziehbaren Abbruch.</p>

<p>Die normalisierten Dateien erhalten erneut Hashes. Zu diesem Zeitpunkt werden der Datenstand und die fünf vorgesehenen Versuchskonfigurationen endgültig festgehalten. Erst nach dieser Abnahme wird die Engine mit realen Untersuchungsdaten gestartet.</p>

### RF-Dateischema und Herkunftsnachweis {#sec-rf-herkunft}


<p>Die fertige Datei enthält die beiden Monatsgrenzen und die daraus berechnete Dezimalrendite. Für jede Zeile wird zusätzlich dokumentiert, welche ursprüngliche Zinsbeobachtung verwendet wurde.</p>


### WEO-Dateimetadaten {#sec-weo-metadaten}


<p>Für jede verwendete Datei werden Ausgabe, Originaldatei, Hash und belegtes Veröffentlichungsdatum gespeichert.</p>
